In [ ]:
# plotting of OD after 5min

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

sns.set(style="whitegrid", font_scale=1.2)

# ---- Load and melt ----
df = pd.read_csv("OD_after_5min_Fe_conc_series.csv")
df_long = df.melt(
    id_vars=["culture", "salt", "conc_uM"],
    value_vars=[c for c in df.columns if c.startswith("OD_after_5min")],
    var_name="replicate",
    value_name="OD"
)


In [ ]:
# df

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

sns.set(style="whitegrid", font_scale=0.2)

# ---- Load and melt ----
df = pd.read_csv("OD_after_5min_Fe_conc_series.csv", keep_default_na=False)
df_long = df.melt(
    id_vars=["culture", "salt", "conc_uM"],
    value_vars=[c for c in df.columns if c.startswith("OD_after_5min")],
    var_name="replicate",
    value_name="OD"
)

order = [0, 10, 20, 40, 80, 100]
df_long["conc_uM"] = df_long["conc_uM"].astype(int)

vals = [df_long.loc[df_long["conc_uM"] == c, "OD"].values for c in order]

# # Iron rust gradient: light tan → deep rust
# colors = {
#     0:   "#D9C5B2",
#     10:  "#C49A72",
#     20:  "#B07040",
#     40:  "#8B4513",
#     80:  "#6B2F0E",
#     100: "#4A1A05",
# }

# ---- Figure ----
fig, ax = plt.subplots(figsize=(2, 1.5), dpi=600)
ax.grid(False)
ax.grid(axis="y", linestyle="-", linewidth=0.8, alpha=0.6)

positions = np.arange(len(order)) + 1.0
rng = np.random.default_rng(7)


# Strip
import matplotlib.colors as mcolors
fill = mcolors.to_rgba("#B5651D", alpha=0.4)   # transparent fill
edge = "#B5651D"                                 # solid edge


for x, y, conc in zip(positions, vals, order):
    jitter = rng.normal(0, 0.06, size=len(y))
    ax.scatter(
        np.full(len(y), x) + jitter,
        y,
        s=60,
        facecolors=fill,
        edgecolors=edge,
        linewidths=1.1,
        zorder=3
        # color= "#B5651D",
        # color=colors[conc],
        # alpha=0.4,
        # zorder=3
    )

# Box
bp = ax.boxplot(
    vals,
    positions=positions,
    widths=0.8,
    patch_artist=True,
    showfliers=False,
    showcaps=False,
    whiskerprops=dict(linewidth=0),
    medianprops=dict(linewidth=1.8, color="dimgray"),
    boxprops=dict(linewidth=1.2, color="dimgray", facecolor="gainsboro", alpha=0.35),
    zorder=1
)

for med in bp["medians"]:
    med.set_zorder(50)

ax.set_xticks(positions)
ax.set_xticklabels(order, rotation=0)
ax.set_xlim(positions[0] - 0.55, positions[-1] + 0.55)
ax.set_xlabel("Fe concentration (µM)")
ax.set_ylabel("OD600 after 5 min settling")
# ax.set_title("FJ28 — Fe concentration series", fontsize=11, pad=6)
ax.margins(x=0.01)

plt.tight_layout(pad=0.3)
plt.savefig("od_fe_conc_stripbox.png", dpi=600, bbox_inches="tight")
plt.show()

In [ ]:
# final stats for paper

In [ ]:
# --- Statistical analysis for Fe concentration series ---
# Note: 6 replicates per concentration (3 bio × 2 tech, pairing unknown)
# ANOVA with Dunnett's post-hoc, all concentrations vs 0 µM control

import pandas as pd
import numpy as np
from statsmodels.formula.api import ols
import statsmodels.api as sm
from scipy.stats import dunnett

df = pd.read_csv('OD_after_5min_Fe_conc_series.csv')

# Melt wide to long
od_cols = [c for c in df.columns if c.startswith('OD_after')]
df_long = df.melt(id_vars=['culture', 'salt', 'conc_uM'], 
                  value_vars=od_cols, 
                  var_name='rep', value_name='OD')

# One-way ANOVA
model = ols('OD ~ C(conc_uM)', data=df_long).fit()
anova_table = sm.stats.anova_lm(model, typ=2)
print("--- ANOVA (Fe concentration series) ---")
print(anova_table)

# Dunnett's: all concentrations vs 0 µM
control = df_long[df_long['conc_uM'] == 0]['OD'].values
compare_concs = [10, 20, 40, 80, 100]
treatments = {c: df_long[df_long['conc_uM'] == c]['OD'].values
              for c in compare_concs}

result = dunnett(*treatments.values(), control=control)
print("\n--- Dunnett's test vs 0 µM ---")
for conc, p in zip(treatments.keys(), result.pvalue):
    print(f"0 µM vs {conc} µM: p = {p}")